In [1]:
import os
import numpy as np
import torch
from torch.nn.utils.rnn import pack_padded_sequence
import torch.nn as nn
from torch.utils.tensorboard import SummaryWriter

import random
import scipy.stats as stats

from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, auc, precision_recall_curve, average_precision_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
import pandas as pd
from utils import performance_dict, sigmoid
from tqdm import tqdm

file = '/home/server/Projects/data/AKI/lstm_trainable.pkl'
df = pd.read_pickle(file)
output_pkl = '/home/server/Projects/data/AKI/results/tabular_combined.pkl'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
class bootstrap_split(object):
    def __init__(self, df):
        self.df = df
        self.i_df = 0   # cycling index out of five
        self.i = 0      # total index out of 25
        self.df_fifths = [] 

    def __iter__(self):
        return self
    def __next__(self):
        return self.next()
    
    def df_to_tensors(self, df):
        remove_cols = ['op_id', 'time_tensors', 'seq_len', 'aki', 'aki_boolean']
        # remove_cols += [col for col in df.columns if '_isna' in col]
        X_tab = torch.tensor(df.drop(columns=remove_cols).values).float()
        X_time = torch.stack(df['time_tensors'].tolist()).float()
        y = torch.tensor(df['aki'].values).unsqueeze(1).float()
        y_binary = df['aki_boolean'].values
        sequence_lengths = df['seq_len'].tolist()
        return X_tab, X_time, sequence_lengths, y, y_binary.astype(bool)
    
    def upsample(self, df):
        df_majority = df[df["aki_boolean"] == 0]
        df_minority = df[df["aki_boolean"] == 1]

        df_minority_upsampled = resample(df_minority,
                    replace=True,  # Sample with replacement
                    n_samples=len(df_majority),  # Match majority class size
                    random_state=42)

        # Combine upsampled minority with majority class
        df_balanced = pd.concat([df_majority, df_minority_upsampled])
        df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
        return df_balanced

    def next(self):
        if self.i == 25:
            raise StopIteration()
        elif self.i % 5 == 0:
            self.i_df = 0
            self.df_fifths = [] 
            df_remainder = self.df
            for remaining_fifths in range(5, 1, -1):
                df_remainder, df_fifth = train_test_split(df_remainder, 
                                            test_size=(1.0/remaining_fifths), 
                                            random_state=42 + (self.i // 5), 
                                            stratify=df_remainder["aki_boolean"])
                self.df_fifths.append(df_fifth)
            self.df_fifths.append(df_remainder)
        df_temp = self.df_fifths.pop(self.i_df)
        # (X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test) == test
        test = self.df_to_tensors(df_temp)
        train = self.df_to_tensors(self.upsample(pd.concat(self.df_fifths)))
        self.df_fifths.insert(self.i_df, df_temp)
        self.i_df += 1
        self.i += 1
        return test, train

def print_confidence_intervals(df_results):
    if 'y_pred_binary' in df_results.columns:
        df_results = df_results.drop(columns=['y_pred_binary'])
    if 'y_prob' in df_results.columns:
        df_results = df_results.drop(columns=['y_prob'])
    lows = []
    means = []
    highs = []
    for col in df_results.columns:
        mean = np.mean(df_results[col].values)
        sem = stats.sem(df_results[col].values)  # Standard error of the mean

        # Get 95% confidence interval
        confidence = 0.95
        n = len(df_results[col].values)
        dof = n - 1  # Degrees of freedom
        ci = stats.t.interval(confidence, dof, loc=mean, scale=sem)
        lows.append(ci[0])
        means.append(mean)
        highs.append(ci[1])
    for col in df_results.columns:
        print(col)
    print()
    for arr in [lows, means, highs]:
        for num in arr:
            print(f"{num:.4f}")
        print()
        
def save_results(model_name, df_results, output_pkl):
    # make into one row
    df_collapsed = pd.DataFrame({col: [df_results[col].values] for col in df_results.columns})
    df_collapsed['model_name'] = model_name

    if os.path.exists(output_pkl):
        df_output = pd.read_pickle(output_pkl)
        if df_output.empty:
            df_output = df_collapsed
        else:
            df_output = df_output[df_output['model_name'] != model_name]
        df_output = pd.concat([df_output, df_collapsed], ignore_index=True)
    else:
        df_output = df_collapsed
    df_output.to_pickle(output_pkl)

In [6]:
def set_seed(seed=42):
    random.seed(seed)                           # Python random module
    np.random.seed(seed)                        # NumPy
    torch.manual_seed(seed)                     # PyTorch CPU
    torch.cuda.manual_seed(seed)                # PyTorch GPU
    torch.cuda.manual_seed_all(seed)            # For multi-GPU setups
    torch.backends.cudnn.deterministic = True   # Makes cuDNN deterministic
    torch.backends.cudnn.benchmark = False      # Avoids dynamic optimizations



class HybridLSTM(nn.Module):
    def __init__(self, num_classes, lstm_width, lstm_hidden, num_layers):
        super(HybridLSTM, self).__init__()
        self.num_classes = num_classes #number of classes
        self.num_layers = num_layers #number of layers
        self.lstm_width = lstm_width #input size
        self.lstm_hidden = lstm_hidden #hidden state
        self.fc_interior = (lstm_hidden) // 2

        self.lstm = nn.LSTM(input_size=lstm_width, hidden_size=lstm_hidden,
                          num_layers=num_layers, batch_first=True) #lstm
        self.fc0 =  nn.Linear(lstm_hidden, self.fc_interior) #fully connected 1
        self.fc = nn.Linear(self.fc_interior, num_classes) #fully connected last layer


        self.relu = torch.tanh #lol
    
    def forward(self,seq_x, seq_len):
        # h_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #hidden state
        # c_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #internal state
        # Propagate input through LSTM
        packed_input = pack_padded_sequence(seq_x, seq_len, batch_first=True, enforce_sorted=False)
        output, (hn, cn) = self.lstm(packed_input)#, (h_0, c_0)) #lstm with input, hidden, and internal state
        hn = hn.view(-1, self.lstm_hidden) #reshaping the data for Dense layer next
        lstm_out = self.relu(hn)

        out = lstm_out
        
        out = self.fc0(out) #first Dense
        out = self.relu(out) #relu
        out = self.fc(out) #Final Output
        return out

In [7]:
# -------------------- MIXED_LSTM CLASSIFICATION --------------------

df_results = pd.DataFrame()
extra_string_i = 0
for test, train in tqdm(bootstrap_split(df)):
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = test
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = train


    num_epochs = 500 #1000 epochs
    learning_rate = 0.001 #0.001 lr

    lstm_width = X_time_train.shape[2] # 24 features

    lstm_hidden = 8 #number of features in hidden state

    num_layers = 1 #number of stacked lstm layers
    num_classes = 1 #number of output classes 

    batch_size = 4000
    num_train_samples = X_tab_train.shape[0]
    num_batches = int(np.ceil(num_train_samples / batch_size))

    extra_string = f'hybrid{extra_string_i}_hs{lstm_hidden}_m'
    extra_string_i += 1
    writer = SummaryWriter('/home/server/Projects/data/AKI/runs/bootstrap/' + extra_string)

    set_seed(42)
    model = HybridLSTM(num_classes, lstm_width, lstm_hidden, num_layers).to(device)

    criterion = torch.nn.MSELoss()    # mean-squared error for regression
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) 
    for epoch in tqdm(range(num_epochs)):
        batch_loss = 0
        for i in range(num_batches):
            beginning_idx = i * batch_size
            ending_idx = min(num_train_samples, (i + 1) * batch_size)
            X_time = X_time_train[beginning_idx: ending_idx].to(device)
            y = y_train[beginning_idx: ending_idx].to(device)
            seq_len = seq_len_train[beginning_idx: ending_idx]
            outputs = model.forward(X_time, seq_len) #forward pass
            
            optimizer.zero_grad()
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            batch_loss += loss.item()


        if epoch % 50 == 49:
            model.eval()
            with torch.no_grad():
                outputs = model(X_time_test.to(device), seq_len_test)
                leg_y_test = (y_test > 0.3).T.tolist()[0]
                leg_y_pred = (outputs > 0.3).T.tolist()[0]
                leg_y_prob = nn.functional.sigmoid(outputs - 0.3).T.tolist()[0]
                dic = performance_dict(leg_y_test, leg_y_pred, leg_y_prob)
                writer.add_scalar('training loss',
                                    batch_loss / num_batches,
                                    epoch * num_batches + i)
                writer.add_scalar('Precision',
                                    dic['Precision'],
                                    epoch * num_batches + i)
                writer.add_scalar('Sensitivity',
                                    dic['Sensitivity'],
                                    epoch * num_batches + i)
                writer.add_scalar('Accuracy',
                                    dic['Accuracy'],
                                    epoch * num_batches + i)
                writer.add_scalar('rc_auc',
                                    dic['rc_auc'],
                                    epoch * num_batches + i)
                writer.add_scalar('pr_auc',
                                    dic['pr_auc'],
                                    epoch * num_batches + i)
                writer.add_scalar('Specificity',
                                    dic['Specificity'],
                                    epoch * num_batches + i)
                writer.add_scalar('Negative Predictive Value',
                                    dic['Negative Predictive Value'],
                                    epoch * num_batches + i)
                writer.add_scalar('F1 Score',
                                    dic['F1 Score'],
                                    epoch * num_batches + i)
            model.train()
    outputs = model(X_time_test.to(device), seq_len_test)
    y_pred = (outputs > 0.3).T.tolist()[0]
    y_prob = nn.functional.sigmoid(outputs - 0.3).T.tolist()[0]
    output_dict = performance_dict(y_binary_test, y_pred, y_prob)

    output_dict['y_pred_binary'] = y_pred
    output_dict['y_prob'] = y_prob
    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

    # in case of outage, since this run takes so long
    temp_file = '/home/server/Projects/data/AKI/bootstrap/temp_lstm.csv'
    df_results.to_csv(temp_file)

    del model
        
save_results('lstm', df_results, output_pkl)

0it [00:00, ?it/s]

100%|██████████| 500/500 [06:50<00:00,  1.22it/s]
25it [3:06:21, 447.27s/it]


In [3]:
def set_seed(seed=42):
    random.seed(seed)                           # Python random module
    np.random.seed(seed)                        # NumPy
    torch.manual_seed(seed)                     # PyTorch CPU
    torch.cuda.manual_seed(seed)                # PyTorch GPU
    torch.cuda.manual_seed_all(seed)            # For multi-GPU setups
    torch.backends.cudnn.deterministic = True   # Makes cuDNN deterministic
    torch.backends.cudnn.benchmark = False      # Avoids dynamic optimizations



class HybridLSTM(nn.Module):
    def __init__(self, num_classes, lstm_width, lstm_hidden, num_layers, tabular_width, mlp_dims=[64, 32, 16]):
        super(HybridLSTM, self).__init__()
        self.num_classes = num_classes #number of classes
        self.num_layers = num_layers #number of layers
        self.lstm_width = lstm_width #input size
        self.lstm_hidden = lstm_hidden #hidden state
        self.tabular_width = tabular_width
        self.mlp_dims = mlp_dims
        self.fc_interior = (lstm_hidden + mlp_dims[-1]) // 2

        self.mlp = self.build_mlp(tabular_width, mlp_dims)

        self.lstm = nn.LSTM(input_size=lstm_width, hidden_size=lstm_hidden,
                          num_layers=num_layers, batch_first=True) #lstm
        self.fc0 =  nn.Linear(lstm_hidden + mlp_dims[-1], self.fc_interior) #fully connected 1
        self.fc = nn.Linear(self.fc_interior, num_classes) #fully connected last layer


        self.relu = torch.tanh #lol
    # def _init_weights(self):
    #     for name, param in self.lstm.named_parameters():
    #         if 'weight' in name:
    #             nn.init.xavier_uniform_(param)  # Use Xavier initialization
    #         elif 'bias' in name:
    #             nn.init.constant_(param, 0)
    def build_mlp(self, tabular_width, hidden_layer_sizes):
        mlp = nn.Sequential()
        prev = hidden_layer_sizes[0]
        mlp.append(nn.Linear(tabular_width, prev))
        for dim in hidden_layer_sizes[1:]:
                mlp.append(nn.ReLU())
                mlp.append(nn.Linear(prev, dim))
                prev = dim
        # mlp.append(nn.Sigmoid()) #if using bceloss
        return mlp
    
    def forward(self,tab_x, seq_x, seq_len):
        # h_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #hidden state
        # c_0 = Variable(torch.zeros(self.num_layers, x.size(0), self.hidden_size)) #internal state
        # Propagate input through LSTM
        packed_input = pack_padded_sequence(seq_x, seq_len, batch_first=True, enforce_sorted=False)
        output, (hn, cn) = self.lstm(packed_input)#, (h_0, c_0)) #lstm with input, hidden, and internal state
        hn = hn.view(-1, self.lstm_hidden) #reshaping the data for Dense layer next
        lstm_out = self.relu(hn)
        mlp_out = self.mlp(tab_x)

        out = torch.cat((lstm_out, mlp_out), dim=1)
        
        out = self.fc0(out) #first Dense
        out = self.relu(out) #relu
        out = self.fc(out) #Final Output
        return out

In [ ]:
# -------------------- MIXED_LSTM CLASSIFICATION --------------------

df_results = pd.DataFrame()
extra_string_i = 0
for test, train in tqdm(bootstrap_split(df)):
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = test
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = train

    num_epochs = 500 #1000 epochs
    learning_rate = 0.001 #0.001 lr

    lstm_width = X_time_train.shape[2] # 24 features
    mlp_width = X_tab_train.shape[1] # 271 without isna

    mlp_dims = [4, 4, 4, 4] #3 layer mlp
    lstm_hidden = 8 #number of features in hidden state

    num_layers = 1 #number of stacked lstm layers
    num_classes = 1 #number of output classes 

    batch_size = 4000
    num_train_samples = X_tab_train.shape[0]
    num_batches = int(np.ceil(num_train_samples / batch_size))

    extra_string = f'long{extra_string_i}_hs{lstm_hidden}_m' + "_".join([str(dim) for dim in mlp_dims])
    extra_string_i += 1
    writer = SummaryWriter('/home/server/Projects/data/AKI/runs/bootstrap/' + extra_string)

    set_seed(42)
    model = HybridLSTM(num_classes, lstm_width, lstm_hidden, num_layers, mlp_width, mlp_dims).to(device)

    criterion = torch.nn.MSELoss()    # mean-squared error for regression
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) 
    for epoch in tqdm(range(num_epochs)):
        batch_loss = 0
        for i in range(num_batches):
            beginning_idx = i * batch_size
            ending_idx = min(num_train_samples, (i + 1) * batch_size)
            X_time = X_time_train[beginning_idx: ending_idx].to(device)
            X_tab = X_tab_train[beginning_idx: ending_idx].to(device)
            y = y_train[beginning_idx: ending_idx].to(device)
            seq_len = seq_len_train[beginning_idx: ending_idx]
            outputs = model.forward(X_tab, X_time, seq_len) #forward pass
            
            optimizer.zero_grad()
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            batch_loss += loss.item()


        if epoch % 50 == 49:
            model.eval()
            with torch.no_grad():
                outputs = model(X_tab_test.to(device), X_time_test.to(device), seq_len_test)
                leg_y_test = (y_test > 0.3).T.tolist()[0]
                leg_y_pred = (outputs > 0.3).T.tolist()[0]
                leg_y_prob = nn.functional.sigmoid(outputs - 0.3).T.tolist()[0]
                dic = performance_dict(leg_y_test, leg_y_pred, leg_y_prob)
                writer.add_scalar('training loss',
                                    batch_loss / num_batches,
                                    epoch * num_batches + i)
                writer.add_scalar('Precision',
                                    dic['Precision'],
                                    epoch * num_batches + i)
                writer.add_scalar('Sensitivity',
                                    dic['Sensitivity'],
                                    epoch * num_batches + i)
                writer.add_scalar('Accuracy',
                                    dic['Accuracy'],
                                    epoch * num_batches + i)
                writer.add_scalar('rc_auc',
                                    dic['rc_auc'],
                                    epoch * num_batches + i)
                writer.add_scalar('pr_auc',
                                    dic['pr_auc'],
                                    epoch * num_batches + i)
                writer.add_scalar('Specificity',
                                    dic['Specificity'],
                                    epoch * num_batches + i)
                writer.add_scalar('Negative Predictive Value',
                                    dic['Negative Predictive Value'],
                                    epoch * num_batches + i)
                writer.add_scalar('F1 Score',
                                    dic['F1 Score'],
                                    epoch * num_batches + i)
            model.train()
    outputs = model(X_tab_test.to(device), X_time_test.to(device), seq_len_test)
    y_pred = (outputs > 0.3).T.tolist()[0]
    y_prob = nn.functional.sigmoid(outputs - 0.3).T.tolist()[0]
    output_dict = performance_dict(y_binary_test, y_pred, y_prob)

    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

    # in case of outage, since this run takes so long
    temp_file = '/home/server/Projects/data/AKI/bootstrap/temp_lstm.csv'
    df_results.to_csv(temp_file)

    del model
        
print_confidence_intervals(df_results)

In [ ]:
# -------------------- MIXED_LSTM CLASSIFICATION --------------------

df_results = pd.DataFrame()
extra_string_i = 0
for test, train in tqdm(bootstrap_split(df)):
    X_tab_test, X_time_test, seq_len_test, y_test, y_binary_test = test
    X_tab_train, X_time_train, seq_len_train, y_train, y_binary_train = train

    num_epochs = 500 #1000 epochs
    learning_rate = 0.001 #0.001 lr

    lstm_width = X_time_train.shape[2] # 24 features
    mlp_width = X_tab_train.shape[1] # 271 without isna

    mlp_dims = [8, 8, 4, 32, 2] #3 layer mlp
    lstm_hidden = 2 #number of features in hidden state

    num_layers = 1 #number of stacked lstm layers
    num_classes = 1 #number of output classes 

    batch_size = 5000
    num_train_samples = X_tab_train.shape[0]
    num_batches = int(np.ceil(num_train_samples / batch_size))

    extra_string = f'{extra_string_i}_hs{lstm_hidden}_m' + "_".join([str(dim) for dim in mlp_dims])
    extra_string_i += 1
    writer = SummaryWriter('/home/server/Projects/data/AKI/runs/bootstrap/' + extra_string)

    set_seed(42)
    model = HybridLSTM(num_classes, lstm_width, lstm_hidden, num_layers, mlp_width, mlp_dims).to(device)

    criterion = torch.nn.MSELoss()    # mean-squared error for regression
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate) 
    for epoch in tqdm(range(num_epochs)):
        batch_loss = 0
        for i in range(num_batches):
            beginning_idx = i * batch_size
            ending_idx = min(num_train_samples, (i + 1) * batch_size)
            X_time = X_time_train[beginning_idx: ending_idx].to(device)
            X_tab = X_tab_train[beginning_idx: ending_idx].to(device)
            y = y_train[beginning_idx: ending_idx].to(device)
            seq_len = seq_len_train[beginning_idx: ending_idx]
            outputs = model.forward(X_tab, X_time, seq_len) #forward pass
            
            optimizer.zero_grad()
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()
            batch_loss += loss.item()


        if epoch % 50 == 49:
            model.eval()
            with torch.no_grad():
                outputs = model(X_tab_test.to(device), X_time_test.to(device), seq_len_test)
                leg_y_test = (y_test > 0.3).T.tolist()[0]
                leg_y_pred = (outputs > 0.3).T.tolist()[0]
                leg_y_prob = nn.functional.sigmoid(outputs - 0.3).T.tolist()[0]
                dic = performance_dict(leg_y_test, leg_y_pred, leg_y_prob)
                writer.add_scalar('training loss',
                                    batch_loss / num_batches,
                                    epoch * num_batches + i)
                writer.add_scalar('Precision',
                                    dic['Precision'],
                                    epoch * num_batches + i)
                writer.add_scalar('Sensitivity',
                                    dic['Sensitivity'],
                                    epoch * num_batches + i)
                writer.add_scalar('Accuracy',
                                    dic['Accuracy'],
                                    epoch * num_batches + i)
                writer.add_scalar('rc_auc',
                                    dic['rc_auc'],
                                    epoch * num_batches + i)
                writer.add_scalar('pr_auc',
                                    dic['pr_auc'],
                                    epoch * num_batches + i)
                writer.add_scalar('Specificity',
                                    dic['Specificity'],
                                    epoch * num_batches + i)
                writer.add_scalar('Negative Predictive Value',
                                    dic['Negative Predictive Value'],
                                    epoch * num_batches + i)
                writer.add_scalar('F1 Score',
                                    dic['F1 Score'],
                                    epoch * num_batches + i)
            model.train()
    outputs = model(X_tab_test.to(device), X_time_test.to(device), seq_len_test)
    y_pred = (outputs > 0.3).T.tolist()[0]
    y_prob = nn.functional.sigmoid(outputs - 0.3).T.tolist()[0]
    output_dict = performance_dict(y_binary_test, y_pred, y_prob)

    if df_results.empty:
        df_results = pd.DataFrame(columns=output_dict.keys())
    df_results.loc[len(df_results)] = output_dict

    # in case of outage, since this run takes so long
    temp_file = '/home/server/Projects/data/AKI/bootstrap/temp_lstm.csv'
    df_results.to_csv(temp_file)

    del model
        
print_confidence_intervals(df_results)

0it [00:00, ?it/s]